# 02 - Preprocessing Check

Here we test our shared preprocess.py and evaluate.py code on results/train.csv.


In [1]:
# import our shared preprocessing and cv helpers from the project folder
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src import evaluate, preprocess

df, feature_cols = preprocess.load_train_data()
print("train shape:", df.shape)
print(
    "feature groups ->",
    "nominal:", len(preprocess.NOMINAL_COLS),
    "| binary:", len(preprocess.BINARY_COLS),
    "| ordinal:", len(preprocess.ORDINAL_COLS),
    "| numeric:", len(preprocess.NUMERIC_COLS),
)


train shape: (12378, 66)
feature groups -> nominal: 4 | binary: 11 | ordinal: 14 | numeric: 34


## 1. Structural Zeros (Smoking and Drinking)

People who do not smoke or drink have NaN in the amount columns. We set those to 0 first so the median imputer does not give them smoker/drinker numbers.


In [2]:
# compare missing counts before and after filling structural zeros
check_cols = ["cigarettes_per_smoking_day", "drinks_per_drinking_day"]
before = df[check_cols].isna().sum()

filler = preprocess.StructuralZeroFiller()
after_df = filler.fit_transform(df[feature_cols])
after = after_df[check_cols].isna().sum()

summary = pd.DataFrame({
    "missing_before": before,
    "missing_after": after,
    "filled_with_0": before - after,
})
display(summary)


,missing_before,missing_after,filled_with_0
cigarettes_per_smoking_day,9861,41,9820
drinks_per_drinking_day,4362,1102,3260


## 2. Test Preprocessors for All 3 Models

We fit only on the fold 0 training rows and transform the validation rows (no leakage).


In [3]:
# test logreg, rf, and catboost preprocessors on fold 0
splits = evaluate.make_cv_splits(df)
tr_idx, va_idx = splits[0]
X_tr = df.iloc[tr_idx][feature_cols]
X_va = df.iloc[va_idx][feature_cols]

rows = []
for model_type in ["logreg", "rf", "catboost"]:
    prep = preprocess.build_preprocessor(model_type, feature_cols)
    Xt_tr = prep.fit_transform(X_tr)
    Xt_va = prep.transform(X_va)
    if model_type == "catboost":
        cat_cols = prep.named_steps["prep"].cat_cols
        val_na = int(Xt_va[cat_cols].isna().sum().sum())
        note = f"keeps numeric NaN, {len(cat_cols)} cat cols as strings"
    else:
        val_na = int(np.isnan(Xt_va).sum())
        note = "median + missing flags + ohe" + (" + RobustScaler" if model_type == "logreg" else "")
    rows.append({
        "model_type": model_type,
        "train_shape": Xt_tr.shape,
        "val_shape": Xt_va.shape,
        "unexpected_nans": val_na,
        "note": note,
    })

display(pd.DataFrame(rows))


,model_type,train_shape,val_shape,unexpected_nans,note
0,logreg,"(9902, 145)","(2476, 145)",0,median + missing flags + ohe + RobustScaler
1,rf,"(9902, 145)","(2476, 145)",0,median + missing flags + ohe
2,catboost,"(9902, 63)","(2476, 63)",0,"keeps numeric NaN, 15 cat cols as strings"


## 3. Check 5-Fold CV Balance

All 5 folds are stratified by both cycle and target.


In [4]:
# check row count and positive rate in each validation fold
fold_rows = []
for fold, (tr, va) in enumerate(splits, start=1):
    fold_rows.append({
        "fold": fold,
        "train_rows": len(tr),
        "val_rows": len(va),
        "train_pos_pct": round(df.iloc[tr]["target"].mean() * 100, 2),
        "val_pos_pct": round(df.iloc[va]["target"].mean() * 100, 2),
    })

display(pd.DataFrame(fold_rows))


,fold,train_rows,val_rows,train_pos_pct,val_pos_pct
0,1,9902,2476,29.61,29.60
1,2,9902,2476,29.61,29.60
2,3,9902,2476,29.61,29.60
3,4,9903,2475,29.61,29.62
4,5,9903,2475,29.61,29.62
